# Pan-Cancer Pathogenic Germline Variant Carrier Analysis — All of Us v8

## Overview

This notebook documents the germline variant carrier analysis performed using the **All of Us Research Program Controlled Tier v8** and short-read whole-genome sequencing data. It identifies participants carrying qualifying pathogenic variants in a predefined **85-gene cancer predisposition panel**, summarizes overall and gene-specific carrier frequencies, evaluates demographic differences, and generates the corresponding tables and figures.

### Analysis workflow

1. Build the participant-level clinical/demographic cohort from the All of Us v8 CDR.
2. Load the v8 short-read WGS Variant Annotation Table.
3. Restrict variant annotations to 97 prespecified MANE transcript identifiers.
4. Apply the study's prespecified ClinVar-classification and consequence filters.
5. Intersect qualifying variants with the WGS genotype MatrixTable.
6. Restrict the genomic dataset to the 85-gene cancer predisposition panel.
7. Identify participants carrying non-reference qualifying variants.
8. Apply the study-specific downstream gene/variant exclusions.
9. Calculate participant-, variant-, and gene-level carrier summaries.
10. Compare carrier frequencies across demographic groups and generate study figures.

### Important reproducibility and provenance notes

- This analysis depends on **All of Us Controlled Tier v8** resources and cannot be reproduced from this repository alone without authorized data access.
- Participant-level data and notebook outputs are intentionally not included in the public-facing notebook.
- The notebook assumes the same software/environment configuration used in the original All of Us Researcher Workbench analysis.

### Key environment resources used by the original notebook

The executable cells reference Researcher Workbench environment variables including `WORKSPACE_BUCKET`, `CDR_STORAGE_PATH`, `WORKSPACE_CDR`, `GOOGLE_PROJECT`, and `WGS_CLINVAR_SPLIT_HAIL_PATH`.

In [ ]:
#import packages and environment variables
import os
import pandas as pd
import subprocess
import numpy as np
import hail as hl
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportions_ztest
import statsmodels.api as sm
import statsmodels.formula.api as smf
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests
from datetime import datetime

In [ ]:
#save environment variables
bucket = os.getenv("WORKSPACE_BUCKET")
print(bucket)

In [ ]:
# import the genomics bucket environment variable to a separate genomics_path variable 
genomics_path = os.getenv("CDR_STORAGE_PATH")
print(genomics_path)

In [ ]:
# Initialize genomic data uses hg38 as a reference
hl.init(default_reference = "GRCh38")

# Clinical Cohort and Demographic Data

This section constructs the participant-level clinical dataset used throughout the analysis from the **All of Us Research Program Controlled Tier, data release v8**. The query retrieves participant identifiers together with gender, date of birth, race, ethnicity, and sex assigned at birth from the `person` table and maps concept IDs to human-readable concept names.

The demographic summaries below are descriptive quality-control checks of the source cohort before the genomic filtering steps are applied.


In [ ]:
# This query represents dataset "working_DS" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_08947253_person_sql = """
    SELECT
        person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id"""

dataset_08947253_person_df = pd.read_gbq(
    dataset_08947253_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_08947253_person_df.head(5)

In [ ]:
dataset_08947253_person_df.shape

## Gender Distribution

Summarize the number of participants in each gender category in the clinical cohort and visualize the distribution as an initial cohort-level quality-control check.


In [ ]:
# Value count for gender
dataset_08947253_person_df.gender.value_counts()

In [ ]:
# Plot the counts using Seaborn
plt.figure(figsize=(8, 6))
sns.countplot(y='gender', data=dataset_08947253_person_df, palette='viridis')
plt.xlabel('Category')
plt.ylabel('Count')
plt.show()

## Race Distribution

Summarize the self-reported race categories represented in the clinical cohort. These categories are later used to calculate race-specific carrier frequencies and statistical comparisons.


In [ ]:
# Value count for race
dataset_08947253_person_df.race.value_counts()

In [ ]:
# Plot the counts using Seaborn
plt.figure(figsize=(8, 6))
sns.countplot(y='race', data=dataset_08947253_person_df, palette='viridis')
plt.xlabel('Category')
plt.ylabel('Count')
plt.show()

## Ethnicity Distribution

Summarize ethnicity in the clinical cohort. The downstream analysis compares participants recorded as **Hispanic or Latino** with those recorded as **Not Hispanic or Latino**.


In [ ]:
# Value count for ethnicity
dataset_08947253_person_df.ethnicity.value_counts()

In [ ]:
# Plot the counts using Seaborn
plt.figure(figsize=(8, 6))
sns.countplot(y='ethnicity', data=dataset_08947253_person_df, palette='viridis')
plt.xlabel('Category')
plt.ylabel('Count')
plt.show()

## Sex Assigned at Birth Distribution

Summarize sex assigned at birth in the clinical cohort. This variable is retained separately from gender and is used in downstream descriptive carrier-frequency analyses.


In [ ]:
# Value count for sex-at-birth
dataset_08947253_person_df.sex_at_birth.value_counts()

In [ ]:
# Plot the counts using Seaborn
plt.figure(figsize=(8, 6))
sns.countplot(y='sex_at_birth', data=dataset_08947253_person_df, palette='viridis')
plt.xlabel('Category')
plt.ylabel('Count')
plt.show()

## Save and Import the Clinical Dataset for Hail

The participant-level clinical dataset is exported as a tab-separated file, copied to the All of Us workspace bucket, and imported as a keyed Hail Table. The participant identifier (`person_id`) is used as the key so that demographic information can later be joined to WGS samples.

In [ ]:
# save dataframe to a .tsv file in the VM / current analysis environment
dataset_08947253_person_df.to_csv("df_whole_DS_v8.tsv", sep = "\t", index=False)

In [ ]:
# copy file from VM to workspace bucket
!gsutil -m cp "df_whole_DS_v8.tsv" {bucket}/data/

In [ ]:
# check if file is copied to workspace bucket
!gsutil -m ls {bucket}/data/

In [ ]:
dataset_tsv = "gs://fc-secure-76c116cf-c2a3-424e-b3f0-f77d015ce1da/data/df_whole_DS_v8.tsv"

In [ ]:
# Convert tsv file to hail matrix table
dataset_hl = (hl.import_table(dataset_tsv,
                              types={'person_id':hl.tstr},
                              impute=True,
                              key='person_id')
             )

# Genomic Data and Variant Annotation

This section identifies qualifying germline variants from the **All of Us v8 short-read whole-genome sequencing (srWGS) SNP/indel Variant Annotation Table (VAT)**. Variant filtering is performed before intersecting the annotations with participant genotypes.


## Load the All of Us v8 Variant Annotation Table

The v8 VAT is loaded from the Controlled Tier genomic data resources using Hail. The table contains genomic coordinates, reference and alternate alleles, transcript annotations, gene symbols, predicted consequences, ClinVar classifications, and related variant annotations used in the filtering steps below.


In [ ]:
#access variant annotation table
!gsutil -u $GOOGLE_PROJECT -m ls gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/vat/

In [ ]:
# read VAT file
vat_path = "gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/vat/vat_complete.bgz.tsv.gz"
vat_path
vat_table = hl.import_table(
    vat_path, 
    force=True, 
    quote='"', 
    delimiter="\t", 
    force_bgz=True, 
    types={"position": hl.tint, "contig": hl.tstr, "ref_allele": hl.tstr, "alt_allele": hl.tstr}
)

In [ ]:
vat_table.describe()

## Restrict Variant Annotations to MANE Transcripts

The study used **97 prespecified MANE transcript identifiers** to define the transcript annotations of interest across the 85-gene cancer predisposition panel. The exact Ensembl transcript IDs are listed in the following code cell.

The VAT `transcript` field is filtered against this list before ClinVar classification and consequence filtering are applied.


In [ ]:
transcripts_of_interest = [
    "ENST00000279146", "ENST00000389048", "ENST00000257430", "ENST00000675843", "ENST00000307078",
    "ENST00000460680", "ENST00000260947", "ENST00000355112", "ENST00000372037", "ENST00000357654",
    "ENST00000380152", "ENST00000259008", "ENST00000639785", "ENST00000367435", "ENST00000261769",
    "ENST00000257904", "ENST00000228872", "ENST00000313407", "ENST00000440480", "ENST00000304494",
    "ENST00000579755", "ENST00000498907", "ENST00000404276", "ENST00000302763", "ENST00000343455",
    "ENST00000325385", "ENST00000275493", "ENST00000263735", "ENST00000366560", "ENST00000285071",
    "ENST00000341105", "ENST00000487848", "ENST00000370818", "ENST00000651081", "ENST00000300177",
    "ENST00000651154", "ENST00000290295", "ENST00000610977", "ENST00000311189", "ENST00000417302",
    "ENST00000288135", "ENST00000358664", "ENST00000555147", "ENST00000450708", "ENST00000397752",
    "ENST00000394351", "ENST00000352241", "ENST00000231790", "ENST00000233146", "ENST00000265081",
    "ENST00000234420", "ENST00000456914", "ENST00000710952", "ENST00000265433", "ENST00000358273",
    "ENST00000338641", "ENST00000651570", "ENST00000261584", "ENST00000257290", "ENST00000226382",
    "ENST00000265849", "ENST00000440232", "ENST00000320574", "ENST00000357628", "ENST00000589228",
    "ENST00000331920", "ENST00000437951", "ENST00000644628", "ENST00000371953", "ENST00000378823",
    "ENST00000337432", "ENST00000345365", "ENST00000267163", "ENST00000617875", "ENST00000355710",
    "ENST00000675419", "ENST00000264932", "ENST00000301761", "ENST00000375499", "ENST00000367975",
    "ENST00000375549", "ENST00000342988", "ENST00000344626", "ENST00000646693", "ENST00000618915",
    "ENST00000644036", "ENST00000348513", "ENST00000326873", "ENST00000369902", "ENST00000310581",
    "ENST00000258439", "ENST00000269305", "ENST00000298552", "ENST00000219476", "ENST00000256474",
    "ENST00000298139", "ENST00000452863"
]


In [ ]:
transcript_vat_table = vat_table.filter(
    hl.any(lambda t: vat_table.transcript.startswith(t), hl.literal(transcripts_of_interest))
)

## Apply the Prespecified ClinVar Classification Filter

The VAT is restricted to the exact set of ClinVar classification strings listed in the following code cell. The list includes the pathogenic/likely pathogenic classification combinations used in the published analysis, including compound ClinVar labels present in the v8 VAT.

After classification filtering, genomic coordinates and alleles are formatted to match the key structure of the Hail MatrixTable.


In [ ]:
lst = [
    'pathogenic', 
    'likely pathogenic', 
    'likely pathogenic, pathogenic',  
    'likely pathogenic, affects',
    'likely pathogenic, association',
    'likely pathogenic, drug response',
    'likely pathogenic, drug response, not provided',
    'likely pathogenic, not provided',
    'likely pathogenic, other',
    'likely pathogenic, pathogenic, affects',
    'likely pathogenic, pathogenic, association',
    'likely pathogenic, pathogenic, drug response',
    'likely pathogenic, pathogenic, drug response, not provided',
    'likely pathogenic, pathogenic, drug response, other',
    'likely pathogenic, pathogenic, drug response, protective',
    'likely pathogenic, pathogenic, drug response, risk factor',
    'likely pathogenic, pathogenic, not provided',
    'likely pathogenic, pathogenic, other',
    'likely pathogenic, pathogenic, protective',
    'likely pathogenic, pathogenic, risk factor',
    'likely pathogenic, pathogenic, risk factor, not provided',
    'likely pathogenic, risk factor',
    'likely risk allele',
    'pathogenic, association',
    'pathogenic, association, protective',
    'pathogenic, confers sensitivity',
    'pathogenic, drug response',
    'pathogenic, drug response, not provided',
    'pathogenic, drug response, risk factor',
    'pathogenic, drug response, risk factor, protective',
    'pathogenic, not provided',
    'pathogenic, other',
    'pathogenic, protective',
    'pathogenic, protective, other',
    'pathogenic, risk factor',
    'pathogenic, risk factor, not provided',
    'pathogenic, risk factor, other',
    'pathogenic, risk factor, protective',
    'risk factor'
]

In [ ]:
# Filter vat_table to only contain Clinvar classification in lst
filtered_vat_table = transcript_vat_table.filter(hl.set(lst).contains(transcript_vat_table.clinvar_classification))

In [ ]:
# Create locus and alleles to match MatrixTable key structure
filtered_vat_table = filtered_vat_table.annotate(
    locus=hl.locus(filtered_vat_table.contig, filtered_vat_table.position, reference_genome='GRCh38'),
    alleles=[filtered_vat_table.ref_allele, filtered_vat_table.alt_allele]
)

In [ ]:
# Key by locus and alleles
filtered_vat_table = filtered_vat_table.key_by('locus', 'alleles')

In [ ]:
filtered_vat_table.describe()

## Exclude Upstream and Downstream Gene Consequences

Variants annotated only as `upstream_gene_variant` or `downstream_gene_variant` are removed. The remaining filtered VAT is carried forward for intersection with the WGS genotype MatrixTable.


In [ ]:
# List of consequences to exclude
exclude_consequences = ['downstream_gene_variant', 'upstream_gene_variant']

# Filter the table to exclude rows with these consequences
filtered_vat_table = filtered_vat_table.filter(
    ~hl.set(exclude_consequences).contains(filtered_vat_table.consequence)
)

## Export the Filtered Variant Annotation Table

The fully filtered annotation table is converted to pandas and saved as a tab-separated file. This intermediate file provides a tabular record of the variants retained after transcript, ClinVar-classification, and consequence filtering.


In [ ]:
transcript_filtered_vat_table_df = filtered_vat_table.to_pandas()

In [ ]:
transcript_filtered_vat_table_df.head()

In [ ]:
transcript_filtered_vat_table_df.to_csv("transcript_filtered_vat_table_v8.tsv", sep = "\t", index=False)

In [ ]:
# Copy files from VM to workspace bucket
!gsutil -m cp "transcript_filtered_vat_table_v8.tsv" {bucket}/data/

In [ ]:
# Check that file is copied
!gsutil -m ls {bucket}/data/

In [ ]:
transcript_filtered_vat_table_df = pd.read_csv('gs://fc-secure-76c116cf-c2a3-424e-b3f0-f77d015ce1da/data/transcript_filtered_vat_table_v8.tsv', sep = "\t")

In [ ]:
transcript_filtered_vat_table_df['gene_symbol'].unique()

# Construct the Genotype-Level Analysis Dataset

The filtered variant annotations are now intersected with the All of Us WGS genotype data. This step restricts the genomic MatrixTable to eligible participants and qualifying variants before participant metadata and variant annotations are attached.


In [ ]:
#importing CLINVAR hail matrix table
mt_wgs_clinvar_path = os.getenv("WGS_CLINVAR_SPLIT_HAIL_PATH")
mt_wgs_clinvar_path

In [ ]:
#read Hail MatrixTable (load genomic data into hail)
mt = hl.read_matrix_table(mt_wgs_clinvar_path)

In [ ]:
#get number of row/variants and columns/samples of MatrixTable
mt.count()

In [ ]:
#print out fields in MatrixTable
mt.describe()

## Restrict WGS Samples and Variants

Two Hail semi-joins define the analysis subset:

1. **Column restriction:** retain WGS samples represented in the clinical participant dataset.
2. **Row restriction:** retain genomic variants represented in the filtered VAT.

This produces `mt_sub`, the genotype MatrixTable used for the subsequent annotation and carrier-identification steps.


In [ ]:
# Select only the samples in dataset_hl
mt_sub = mt.semi_join_cols(dataset_hl)

In [ ]:
# Select only the variants in filtered_vat_table
mt_sub = mt_sub.semi_join_rows(filtered_vat_table)

In [ ]:
mt_sub.describe()

In [ ]:
# # # DONT RUN, IT TAKES A LONG TIME
# mt_sub.count()

## Annotate Samples and Variants

Participant demographic metadata are attached to MatrixTable columns, while the filtered VAT annotations are attached to MatrixTable rows. The resulting MatrixTable therefore links each retained genotype to both participant-level demographic information and variant-level annotations.


In [ ]:
# Annotate MatrixTable columns with metadata
mt_sub = mt_sub.annotate_cols(metadata=dataset_hl[mt_sub.s])

In [ ]:
# Annotate MatrixTable rows with variant annotations
mt_sub = mt_sub.annotate_rows(annotations=filtered_vat_table[mt_sub.locus, mt_sub.alleles])

In [ ]:
mt_sub.describe()

## Save Participant Metadata for Downstream Analyses

The annotated MatrixTable columns are extracted to a Hail Table and then converted to pandas. This file contains the participant IDs and demographic metadata for the WGS analysis cohort and is reused as the denominator dataset for downstream subgroup analyses.


In [ ]:
# Extract the column fields (metadata and sample ID 's') into a Hail Table
metadata_table = mt_sub.cols()

# Convert the Hail Table to a Pandas DataFrame
metadata_df = metadata_table.to_pandas()

In [ ]:
# Inspect the resulting DataFrame
metadata_df.head()

In [ ]:
# Save the DataFrame to a CSV file
metadata_df.to_csv('metadata_v8.csv', index=False)

In [ ]:
# Copy files from VM to workspace bucket
!gsutil -m cp "metadata_v8.csv" {bucket}/data/

In [ ]:
# Check that file is copied
!gsutil -m ls {bucket}/data/

In [ ]:
metadata_df = pd.read_csv('gs://fc-secure-76c116cf-c2a3-424e-b3f0-f77d015ce1da/data/metadata_v8.csv')
metadata_df.shape

## Restrict to the 85-Gene Cancer Predisposition Panel

The genomic dataset is restricted to the prespecified **85-gene panel** listed in the code cell below. The gene list is retained exactly as used in the published analysis.

After gene filtering, each genotype entry is annotated according to whether the participant carries a non-reference allele.


In [ ]:
# Kind of redundant though
# Filter the MatrixTable to specific genes
genes_of_interest = ['AIP', 'ALK', 'APC', 'ATM', 'AXIN2', 'BAP1', 'BARD1', 'BLM', 'BMPR1A', 'BRCA1', 'BRCA2', 
                     'BRIP1', 'CASR', 'CDC73', 'CDH1', 'CDK4', 'CDKN1B', 'CDKN1C', 'CDKN2A', 'CEBPA', 'CHEK2', 
                     'CTNNA1', 'DICER1', 'DIS3L2', 'EGFR', 'EPCAM', 'FH', 'FLCN', 'GATA2', 'GPC3', 'GREM1', 
                     'HOXB13', 'HRAS', 'KIT', 'MAX', 'MC1R', 'MEN1', 'MET', 'MITF', 'MLH1', 'MSH2', 'MSH3', 
                     'MSH6', 'MUTYH', 'NBN', 'NF1', 'NF2', 'NTHL1', 'PALB2', 'PDGFRA', 'PHOX2B', 'PMS2', 
                     'POLD1', 'POLE', 'POT1', 'PRKAR1A', 'PTCH1', 'PTEN', 'RAD50', 'RAD51C', 'RAD51D', 'RB1', 
                     'RECQL4', 'RET', 'RUNX1', 'SDHA', 'SDHAF2', 'SDHB', 'SDHC', 'SDHD', 'SMAD4', 'SMARCA4', 
                     'SMARCB1', 'SMARCE1', 'STK11', 'SUFU', 'TERC', 'TERT', 'TMEM127', 'TP53', 'TSC1', 'TSC2', 
                     'VHL', 'WRN', 'WT1']

In [ ]:
mt_filtered = mt_sub.filter_rows(hl.literal(genes_of_interest).contains(mt_sub.annotations.gene_symbol))

In [ ]:
# Annotate entries with a flag indicating non-reference variants
mt_filtered = mt_filtered.annotate_entries(has_variant=mt_filtered.GT.is_non_ref())

In [ ]:
mt_filtered.describe()

## Create the Participant–Variant Entries Table

The gene-filtered MatrixTable is converted to an entries table, and only non-reference genotypes are retained. The resulting table contains participant–variant observations together with the demographic and variant annotations needed for downstream carrier and subgroup analyses.


In [ ]:
# Convert MatrixTable to Table for aggregation
entries_table = mt_filtered.entries()

In [ ]:
# Filter entries to include only non-reference variants
entries_table = entries_table.filter(entries_table.has_variant)

In [ ]:
entries_table.describe()

In [ ]:
entries_table_df = entries_table.to_pandas()

# Save to a CSV file (optional)
entries_table_df.to_csv('entries_table_full_v8.csv', index=False)

# Copy files from VM to workspace bucket
!gsutil -m cp "entries_table_full_v8.csv" {bucket}/data/

In [ ]:
# Check that file is copied
!gsutil -m ls {bucket}/data/

# Downstream Carrier Analysis

This section starts from the exported participant–variant table and applies the study-specific downstream exclusions used in the published analysis before calculating carrier frequencies, demographic summaries, and gene- and variant-level counts.

In [ ]:
entries_table_df = pd.read_csv(f'{bucket}/data/entries_table_full_v8.csv')

In [ ]:
entries_table_df.shape

## Apply Study-Specific Downstream Exclusions

Before calculating downstream carrier summaries, the original analysis removes **MC1R** and then applies the explicit gene/variant exclusion rules encoded in the next cell, including EPCAM deletion records and the listed individual variant IDs. This was done based on lab discussion.


In [ ]:
entries_table_df = entries_table_df[entries_table_df['annotations.gene_symbol'] != 'MC1R']
entries_table_df.shape

In [ ]:
entries_table_df = entries_table_df[
    ~(
        ((entries_table_df['annotations.gene_symbol'] == 'EPCAM') & 
         (entries_table_df['annotations.variant_type'] == 'deletion')) |  
        ((entries_table_df['annotations.gene_symbol'] == 'PDGFRA') & 
         (entries_table_df['annotations.vid'] == '4-54281602-C-T')) |  
        ((entries_table_df['annotations.gene_symbol'] == 'EGFR') & 
         (entries_table_df['annotations.vid'] == '7-55173126-T-C')) |  
        ((entries_table_df['annotations.gene_symbol'] == 'RET') & 
         (entries_table_df['annotations.vid'] == '10-43100576-C-T')) |  
        ((entries_table_df['annotations.gene_symbol'] == 'RET') & 
         (entries_table_df['annotations.vid'] == '10-43106497-G-A')) |  
        ((entries_table_df['annotations.gene_symbol'] == 'TSC1') & 
         (entries_table_df['annotations.vid'] == '9-132921940-T-G')) |  
        ((entries_table_df['annotations.gene_symbol'] == 'TMEM127') & 
         (entries_table_df['annotations.vid'] == '2-96265399-G-A')) |  
        ((entries_table_df['annotations.gene_symbol'] == 'TERT') & 
         (entries_table_df['annotations.vid'] == '5-1293489-C-G')) |  
        ((entries_table_df['annotations.gene_symbol'] == 'TERT') & 
         (entries_table_df['annotations.vid'] == '5-1268581-G-A')) |  
        ((entries_table_df['annotations.gene_symbol'] == 'TERT') & 
         (entries_table_df['annotations.vid'] == '5-1254461-C-T')) |  
        ((entries_table_df['annotations.gene_symbol'] == 'POLE') & 
         (entries_table_df['annotations.vid'] == '12-132680048-T-C')) |  
        ((entries_table_df['annotations.gene_symbol'] == 'POLE') & 
         (entries_table_df['annotations.vid'] == '12-132677577-C-T')) |  
        ((entries_table_df['annotations.gene_symbol'] == 'ALK') & 
         (entries_table_df['annotations.vid'] == '2-29220747-C-T'))
    )
]
entries_table_df.shape

In [ ]:
len(entries_table_df['annotations.transcript'].unique())

In [ ]:
entries_table_df.head()

## Inspect the Final Qualifying Variant Set

The next cells inspect the ClinVar classifications represented after all upstream and downstream filters, deduplicate variant IDs, and export a variant-level classification table for downstream reporting.


In [ ]:
entries_table_df['annotations.clinvar_classification'].unique()

In [ ]:
unique_vid = entries_table_df[['annotations.vid', 'annotations.clinvar_classification']].drop_duplicates(subset=['annotations.vid'])

In [ ]:
unique_vid.shape

In [ ]:
unique_vid.to_csv('unique_vid_classification_v8.csv', index=False)

# Copy files from VM to workspace bucket
!gsutil -m cp "unique_vid_classification_v8.csv" {bucket}/data/

## Summarize ClinVar Classifications by Gene and Variant

The filtered participant–variant data are grouped by gene symbol, ClinVar classification, and variant ID. The number of unique participants carrying each variant is then calculated and exported as a summary table.


In [ ]:
# Group by gene and classification, then count unique individuals
grouped = entries_table_df.groupby(['annotations.gene_symbol', 'annotations.clinvar_classification', 'annotations.vid'])['s'].nunique()
gene_classification = grouped.sort_values(ascending=False).reset_index()
gene_classification = gene_classification.rename(columns={
    'annotations.gene_symbol': 'gene_symbol', 
    'annotations.clinvar_classification': 'clinvar_classification',
    's': 'unique_individuals'  # assuming 's' is the count column
})
gene_classification.head()

In [ ]:
# Save to a CSV file (optional)
gene_classification.to_csv('gene_clinvarClassification_v8.csv', index=False)

# Copy files from VM to workspace bucket
!gsutil -m cp "gene_clinvarClassification_v8.csv" {bucket}/data/

# Carrier Counts and Descriptive Summaries

The following analyses summarize qualifying-variant carriers at three complementary levels:

- **Participant level:** each participant is counted once when estimating the overall carrier prevalence.
- **Variant level:** participants are counted for each distinct variant.
- **Gene level:** each participant is counted at most once per gene, even if that participant carries more than one qualifying variant in that gene.

These distinctions are important because a participant can carry qualifying variants in more than one gene.


## Overall Carrier Prevalence

Participants are deduplicated by sample ID so that each carrier contributes only once to the overall prevalence estimate. The code also calculates the age summary for this carrier cohort.

In [ ]:
# Deduplicate to count each individual only once across all 85 genes with pathogenic variants
unique_individuals_with_pathogenic_variant = entries_table_df.drop_duplicates(subset=['s'])

In [ ]:
# Median Age for Carrier Cohort
import scipy.stats as stats
from datetime import datetime

# Convert to datetime
unique_individuals_with_pathogenic_variant['metadata.date_of_birth'] = pd.to_datetime(unique_individuals_with_pathogenic_variant['metadata.date_of_birth'])

# Calculate age
current_year = datetime.now().year
unique_individuals_with_pathogenic_variant['Age'] = current_year - unique_individuals_with_pathogenic_variant['metadata.date_of_birth'].dt.year

# Compute median age
median_age = unique_individuals_with_pathogenic_variant['Age'].median()

# Compute 95% confidence interval
n = len(unique_individuals_with_pathogenic_variant)
se = stats.sem(unique_individuals_with_pathogenic_variant['Age'])  # Standard Error
ci_lower, ci_upper = stats.t.interval(0.95, df=n-1, loc=median_age, scale=se)

# Display results
print(f"Median Age: {median_age:.2f} years")
print(f"95% CI: ({ci_lower:.2f}, {ci_upper:.2f}) years")

In [ ]:
unique_individuals_with_pathogenic_variant.head()

In [ ]:
individuals_with_pathogenic_variant_count = unique_individuals_with_pathogenic_variant.shape[0]
individuals_with_pathogenic_variant_count

In [ ]:
total_population_count = 414830
# Calculate the percentage of individuals with any pathogenic/likely pathogenic variant
percentage_with_pathogenic_variant = (individuals_with_pathogenic_variant_count / total_population_count) * 100

# Step 15: Output the result
print(f"Total Population: {total_population_count}")
print(f"Individuals with any pathogenic/likely pathogenic variant: {individuals_with_pathogenic_variant_count}")
print(f"Percentage of individuals with pathogenic/likely pathogenic variants: {percentage_with_pathogenic_variant:.2f}%")

## Variant-Level Carrier Counts

For variant-level summaries, records are deduplicated by gene, locus, alleles, transcript, and participant ID. Counts are then generated for the demographic categories used in the study, including gender, race, ethnicity, and sex assigned at birth.


In [ ]:
# Deduplicate based on gene_symbol, locus, alleles, transcript, and sample (s)
entries_table_vc_df = entries_table_df.drop_duplicates(subset=['annotations.gene_symbol', 
                                                            'locus', 
                                                            'alleles', 
                                                            'annotations.transcript', 
                                                            's'])


In [ ]:
# Perform the group by operation
variant_counts_df = entries_table_vc_df.groupby(
    ['annotations.gene_symbol', 'locus', 'alleles', 'annotations.transcript']
).agg(
    male_count=('metadata.gender', lambda x: (x == 'Male').sum()),
    female_count=('metadata.gender', lambda x: (x == 'Female').sum()),
    black_African_count=('metadata.race', lambda x: (x == 'Black or African American').sum()),
    white_count=('metadata.race', lambda x: (x == 'White').sum()),
    asian_count=('metadata.race', lambda x: (x == 'Asian').sum()),
    middle_eastern_count=('metadata.race', lambda x: (x == 'Middle Eastern or North African').sum()),
    hawaiian_count=('metadata.race', lambda x: (x == 'Native Hawaiian or Other Pacific Islander').sum()),
    american_indian_alaska_count=('metadata.race', lambda x: (x == 'American Indian or Alaska Native').sum()),
    hispanic_count=('metadata.ethnicity', lambda x: (x == 'Hispanic or Latino').sum()),
    non_hispanic_count=('metadata.ethnicity', lambda x: (x == 'Not Hispanic or Latino').sum()),
    sex_at_birth_male_count=('metadata.sex_at_birth', lambda x: (x == 'Male').sum()),
    sex_at_birth_female_count=('metadata.sex_at_birth', lambda x: (x == 'Female').sum())
).reset_index()


In [ ]:
variant_counts_df['locus'] = variant_counts_df['locus'].astype(str)

In [ ]:
variant_counts_df = variant_counts_df.rename(columns={'annotations.gene_symbol': 'gene_symbol', 'annotations.transcript': 'transcript'})

### Construct a Variant Identifier (`vid`)

The chromosome, genomic position, reference allele, and alternate allele are combined into a standardized identifier of the form:

`chromosome-position-reference-alternate`

This identifier is used in subsequent variant-level summaries and visualizations.


In [ ]:
# Split 'locus' into chromosome and position
variant_counts_df['alleles'] = variant_counts_df['alleles'].apply(lambda x: eval(x) if isinstance(x, str) else x)
variant_counts_df[['chromosome', 'position']] = variant_counts_df['locus'].str.split(':', expand=True)
variant_counts_df['chromosome'] = variant_counts_df['chromosome'].str.replace('chr', '')  # Remove 'chr' prefix

# # Extract 'ref' and 'alt' alleles from the 'alleles' column
variant_counts_df['ref'] = variant_counts_df['alleles'].apply(lambda x: x[0])  # First allele
variant_counts_df['alt'] = variant_counts_df['alleles'].apply(lambda x: x[1])  # Second allele

# # Create the 'vid' column in the format 'chromosome-position-ref-alt'
variant_counts_df['vid'] = variant_counts_df['chromosome'] + '-' + variant_counts_df['position'] + '-' + variant_counts_df['ref'] + '-' + variant_counts_df['alt']

# # Drop intermediate columns if not needed
variant_counts_df = variant_counts_df.drop(columns=['chromosome', 'position', 'ref', 'alt'])
variant_counts_df.head()


## Gene-Level Unique Carrier Counts

For gene-level summaries, records are deduplicated by **gene symbol and participant ID**. A participant with multiple qualifying variants in the same gene is therefore counted once for that gene, while a participant carrying variants in different genes can contribute to more than one gene.

The resulting table contains the total number of unique carriers per gene together with demographic-stratified carrier counts.


In [ ]:
# Remove duplicates to ensure each sample is counted only once per gene. But a person can still have variants in multiple genes
entries_table_dedup_df = entries_table_df.drop_duplicates(subset=['annotations.gene_symbol', 's'])

In [ ]:
entries_table_dedup_df.shape

In [ ]:
# Perform the group-by operation
counts_by_genes_df = entries_table_dedup_df.groupby('annotations.gene_symbol').agg(
    unique_individual_count=('s', 'count'),  # Count unique samples per gene
    male_count=('metadata.gender', lambda x: (x == 'Male').sum()),
    female_count=('metadata.gender', lambda x: (x == 'Female').sum()),
    black_African_count=('metadata.race', lambda x: (x == 'Black or African American').sum()),
    white_count=('metadata.race', lambda x: (x == 'White').sum()),
    asian_count=('metadata.race', lambda x: (x == 'Asian').sum()),
    middle_eastern_count=('metadata.race', lambda x: (x == 'Middle Eastern or North African').sum()),
    hawaiian_count=('metadata.race', lambda x: (x == 'Native Hawaiian or Other Pacific Islander').sum()),
    american_indian_count=('metadata.race', lambda x: (x == 'American Indian or Alaska Native').sum()),
    hispanic_count=('metadata.ethnicity', lambda x: (x == 'Hispanic or Latino').sum()),
    non_hispanic_count=('metadata.ethnicity', lambda x: (x == 'Not Hispanic or Latino').sum()),
    sex_at_birth_male_count=('metadata.sex_at_birth', lambda x: (x == 'Male').sum()),
    sex_at_birth_female_count=('metadata.sex_at_birth', lambda x: (x == 'Female').sum())
).reset_index()


In [ ]:
counts_by_genes_df = counts_by_genes_df.rename(columns={'annotations.gene_symbol': 'gene_symbol'})

In [ ]:
counts_by_genes_df.head()

In [ ]:
counts_by_genes_df['unique_individual_count'].sum()

In [ ]:
counts_by_genes_df.sum()

## Build Demographic Denominators for Table-Level Summaries

The WGS cohort metadata are reloaded and used to define the total population counts for each demographic category. Carrier counts and total counts are then paired so that demographic-specific carrier percentages can be calculated consistently for the tables and plots that follow.


In [ ]:
metadata_df = pd.read_csv('gs://fc-secure-76c116cf-c2a3-424e-b3f0-f77d015ce1da/data/metadata_v8.csv')
metadata_df.head()

In [ ]:
# Median Age for the entire Cohort
from datetime import datetime

# Convert to datetime
metadata_df['metadata.date_of_birth'] = pd.to_datetime(metadata_df['metadata.date_of_birth'])

# Calculate age
current_year = datetime.now().year
metadata_df['Age'] = current_year - metadata_df['metadata.date_of_birth'].dt.year

# Compute median age
median_age = metadata_df['Age'].median()

# Compute 95% confidence interval
n = len(metadata_df)
se = stats.sem(metadata_df['Age'])  # Standard Error
ci_lower, ci_upper = stats.t.interval(0.95, df=n-1, loc=median_age, scale=se)

# Display results
print(f"Median Age: {median_age:.2f} years")
print(f"95% CI: ({ci_lower:.2f}, {ci_upper:.2f}) years")

In [ ]:
metadata_df.shape

In [ ]:
metadata_df['metadata.gender'].value_counts()

In [ ]:
carrier_population = {
    'male': unique_individuals_with_pathogenic_variant[unique_individuals_with_pathogenic_variant['metadata.gender'] == 'Male'].shape[0],
    'female': unique_individuals_with_pathogenic_variant[unique_individuals_with_pathogenic_variant['metadata.gender'] == 'Female'].shape[0],
    'black_African': unique_individuals_with_pathogenic_variant[unique_individuals_with_pathogenic_variant['metadata.race'] == 'Black or African American'].shape[0],
    'white': unique_individuals_with_pathogenic_variant[unique_individuals_with_pathogenic_variant['metadata.race'] == 'White'].shape[0],
    'asian': unique_individuals_with_pathogenic_variant[unique_individuals_with_pathogenic_variant['metadata.race'] == 'Asian'].shape[0],
    'middle_eastern': unique_individuals_with_pathogenic_variant[unique_individuals_with_pathogenic_variant['metadata.race'] == 'Middle Eastern or North African'].shape[0],
    'hawaiian': unique_individuals_with_pathogenic_variant[unique_individuals_with_pathogenic_variant['metadata.race'] == 'Native Hawaiian or Other Pacific Islander'].shape[0],
    'american_indian': unique_individuals_with_pathogenic_variant[unique_individuals_with_pathogenic_variant['metadata.race'] == 'American Indian or Alaska Native'].shape[0],
    'hispanic': unique_individuals_with_pathogenic_variant[unique_individuals_with_pathogenic_variant['metadata.ethnicity'] == 'Hispanic or Latino'].shape[0],
    'non_hispanic': unique_individuals_with_pathogenic_variant[unique_individuals_with_pathogenic_variant['metadata.ethnicity'] == 'Not Hispanic or Latino'].shape[0],
    'sex_at_birth_male': unique_individuals_with_pathogenic_variant[unique_individuals_with_pathogenic_variant['metadata.sex_at_birth'] == 'Male'].shape[0],
    'sex_at_birth_female': unique_individuals_with_pathogenic_variant[unique_individuals_with_pathogenic_variant['metadata.sex_at_birth'] == 'Female'].shape[0],
}

In [ ]:
carrier_population

In [ ]:
total_population = {
    'male': metadata_df[metadata_df['metadata.gender'] == 'Male'].shape[0],
    'female': metadata_df[metadata_df['metadata.gender'] == 'Female'].shape[0],
    'black_African': metadata_df[metadata_df['metadata.race'] == 'Black or African American'].shape[0],
    'white': metadata_df[metadata_df['metadata.race'] == 'White'].shape[0],
    'asian': metadata_df[metadata_df['metadata.race'] == 'Asian'].shape[0],
    'middle_eastern': metadata_df[metadata_df['metadata.race'] == 'Middle Eastern or North African'].shape[0],
    'hawaiian': metadata_df[metadata_df['metadata.race'] == 'Native Hawaiian or Other Pacific Islander'].shape[0],
    'american_indian': metadata_df[metadata_df['metadata.race'] == 'American Indian or Alaska Native'].shape[0],
    'hispanic': metadata_df[metadata_df['metadata.ethnicity'] == 'Hispanic or Latino'].shape[0],
    'non_hispanic': metadata_df[metadata_df['metadata.ethnicity'] == 'Not Hispanic or Latino'].shape[0],
    'sex_at_birth_male': metadata_df[metadata_df['metadata.sex_at_birth'] == 'Male'].shape[0],
    'sex_at_birth_female': metadata_df[metadata_df['metadata.sex_at_birth'] == 'Female'].shape[0],
}

In [ ]:
total_population

In [ ]:
# Percentages for Table 1:
# Calculate percentage of carriers in each group
percentage_population = {}

for key in carrier_population:
    total = total_population.get(key, 0)  # Get total count, default to 0 if not found
    carrier = carrier_population.get(key, 0)  # Get carrier count, default to 0 if not found
    
    # Avoid division by zero
    if total > 0:
        percentage_population[key] = (carrier / total) * 100
    else:
        percentage_population[key] = None  # Set to None if total is zero

# Display the result
for key, value in percentage_population.items():
    print(f"{key}: {value:.2f}%" if value is not None else f"{key}: No data available")


In [ ]:
# Step 1: Define a function to calculate percentage based on demographic category and total population counts
def calculate_percentage(row, category):
    # Gender-based percentages
    if row[category] == 'Male':
        return (row['count'] / total_population['male']) * 100
    elif row[category] == 'Female':
        return (row['count'] / total_population['female']) * 100
    
    # Ethnicity-based percentages
    elif row[category] == 'Hispanic':
        return (row['count'] / total_population['hispanic']) * 100
    elif row[category] == 'Non-Hispanic':
        return (row['count'] / total_population['non_hispanic']) * 100
    
    # Race-based percentages
    elif row[category] == 'White':
        return (row['count'] / total_population['white']) * 100
    elif row[category] == 'Black or African American':
        return (row['count'] / total_population['black_African']) * 100
    elif row[category] == 'Asian':
        return (row['count'] / total_population['asian']) * 100
    elif row[category] == 'Native Hawaiian or Other Pacific Islander':
        return (row['count'] / total_population['hawaiian']) * 100
    elif row[category] == 'Middle Eastern or North African':
        return (row['count'] / total_population['middle_eastern']) * 100
    elif row[category] == 'American Indian or Alaska Native':
        return (row['count'] / total_population['american_indian']) * 100
    
    
    # Sex-at-birth percentages
    elif row[category] == 'Male at Birth' and category == 'sex_at_birth':
        return (row['count'] / total_population['sex_at_birth_male']) * 100
    elif row[category] == 'Female at Birth' and category == 'sex_at_birth':
        return (row['count'] / total_population['sex_at_birth_female']) * 100
    
    return row['count']  # For any other categories, return raw count


# Statistical Comparisons

Statistical tests are used to compare carrier frequencies across demographic groups and to compare the age distributions of carriers and non-carriers. This section first performs cohort-wide comparisons and then repeats selected comparisons on a gene-by-gene basis.

## Cohort-Wide Group Comparisons

The tests in this subsection evaluate whether the overall proportion of participants carrying at least one qualifying variant differs across demographic groups, or whether carrier and non-carrier age distributions differ.


### Gender

A two-sided **two-proportion z-test** compares the overall carrier proportion between participants recorded as male and female.


In [ ]:
# Data
carrier_males = carrier_population['male']
total_males = total_population['male']
carrier_females = carrier_population['female']
total_females = total_population['female']

# Proportions
p1 = carrier_males / total_males  # Male carrier rate
p2 = carrier_females / total_females  # Female carrier rate

# Two-Proportion Z-Test
counts = np.array([carrier_males, carrier_females])  # Number of carriers
nobs = np.array([total_males, total_females])  # Total individuals in each group

z_stat, p_value = proportions_ztest(counts, nobs, alternative='two-sided')

# Output results
print(f"Two-Proportion Z-Test Statistic: {z_stat:.4f}")
print(f"P-Value: {p_value:.4e}")  # Scientific notation for readability


### Ethnicity

A two-sided **two-proportion z-test** compares the overall carrier proportion between participants recorded as Hispanic or Latino and those recorded as Not Hispanic or Latino.


In [ ]:

# Data
carrier_hispanic = carrier_population['hispanic']
total_hispanic = total_population['hispanic']
carrier_non_hispanic = carrier_population['non_hispanic']
total_non_hispanic = total_population['non_hispanic']

# Proportions
p1 = carrier_hispanic / total_hispanic  
p2 = carrier_non_hispanic / total_non_hispanic  # Female carrier rate

# Two-Proportion Z-Test
counts = np.array([carrier_hispanic, carrier_non_hispanic])  # Number of carriers
nobs = np.array([total_hispanic, total_non_hispanic])  # Total individuals in each group

z_stat, p_value = proportions_ztest(counts, nobs, alternative='two-sided')

# Output results
print(f"Two-Proportion Z-Test Statistic: {z_stat:.4f}")
print(f"P-Value: {p_value:.4e}")  # Scientific notation for readability


In [ ]:
(carrier_non_hispanic/total_non_hispanic)*100

### Race

A contingency table is constructed from carrier and non-carrier counts across the six race categories included in the analysis. A **chi-square test of independence** is then used to evaluate whether overall carrier status differs across these categories.


In [ ]:
import scipy.stats as stats
import numpy as np

# Define race categories
race_categories = [
    "white", "asian", "black_African", "middle_eastern", "hawaiian", "american_indian"
]

# Retrieve total population counts
total_population_counts = [total_population[race] for race in race_categories]

# Retrieve carrier counts from counts_by_genes_df
carrier_counts = [carrier_population[race] for race in race_categories]

# Compute non-carrier counts
non_carrier_counts = [total - carriers for total, carriers in zip(total_population_counts, carrier_counts)]

# Create contingency table (each row represents a race)
observed = np.array([
    carrier_counts,  # Carriers per race
    non_carrier_counts  # Non-Carriers per race
]).T  # Transpose to match (races x 2) format

# Perform Chi-Square Test
chi2_stat, chi2_p_value, chi2_dof, chi2_expected = stats.chi2_contingency(observed)

# Output results
chi2_stat, chi2_p_value


## Gene-by-Gene Statistical Analysis

The demographic comparisons are next repeated separately for each gene. Where implemented, raw gene-level P values are adjusted using the **Bonferroni method** to account for multiple gene-level comparisons.


### Gene-by-Gender Comparison

For each gene, a two-sided two-proportion z-test compares the carrier proportion between male and female participants. Bonferroni-adjusted P values are calculated across genes, and genes meeting the adjusted significance threshold are flagged.


In [ ]:
# Re-import necessary libraries
import pandas as pd
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns

data = pd.DataFrame({
    'Gene': counts_by_genes_df['gene_symbol'],
    'Male_Carriers': counts_by_genes_df['male_count'],  
    'Female_Carriers': counts_by_genes_df['female_count'],
    'Total_Males': [total_population['male']] * len(counts_by_genes_df['gene_symbol']),  # Keep total male count the same for all genes
    'Total_Females': [total_population['female']] * len(counts_by_genes_df['gene_symbol'])  # Keep total female count the same for all genes
})

# Compute non-carriers
data['Male_Non_Carriers'] = data['Total_Males'] - data['Male_Carriers']
data['Female_Non_Carriers'] = data['Total_Females'] - data['Female_Carriers']

# Function to perform two-proportion Z-test for each gene
def two_proportion_z_test(row):
    counts = np.array([row['Male_Carriers'], row['Female_Carriers']])
    nobs = np.array([row['Total_Males'], row['Total_Females']])
    z_stat, p_value = proportions_ztest(counts, nobs, alternative='two-sided')
    return p_value

# Apply two-proportion Z-test to all genes
data['Z_Test_P_Value'] = data.apply(two_proportion_z_test, axis=1)

# Apply Bonferroni correction for Z-test
data['Z_Test_Adjusted_P_Value'] = multipletests(data['Z_Test_P_Value'], method='bonferroni')[1]

# Identify significant genes after Z-test correction
data['Z_Test_Significant'] = data['Z_Test_Adjusted_P_Value'] < 0.05

# Sort by adjusted p-value
data = data.sort_values(by='Z_Test_Adjusted_P_Value')
data.head()

### Gene-by-Ethnicity Comparison

For each gene, a two-sided two-proportion z-test compares carrier proportions between Hispanic/Latino and non-Hispanic/Latino participants. Bonferroni-adjusted P values are used to identify statistically significant gene-level differences.


In [ ]:
data = pd.DataFrame({
    'Gene': counts_by_genes_df['gene_symbol'],
    'hispanic_Carriers': counts_by_genes_df['hispanic_count'],  
    'non_hispanic_Carriers': counts_by_genes_df['non_hispanic_count'],
    'Total_Hispanic': [total_population['hispanic']] * len(counts_by_genes_df['gene_symbol']),  # Keep total male count the same for all genes
    'Total_Non_Hispanic': [total_population['non_hispanic']] * len(counts_by_genes_df['gene_symbol'])  # Keep total female count the same for all genes
})

# Compute non-carriers
data['Hispanic_Non_Carriers'] = data['Total_Hispanic'] - data['hispanic_Carriers']
data['Non_Hispanic_Non_Carriers'] = data['Total_Non_Hispanic'] - data['non_hispanic_Carriers']

# Function to perform two-proportion Z-test for each gene
def two_proportion_z_test(row):
    counts = np.array([row['hispanic_Carriers'], row['non_hispanic_Carriers']])
    nobs = np.array([row['Total_Hispanic'], row['Total_Non_Hispanic']])
    z_stat, p_value = proportions_ztest(counts, nobs, alternative='two-sided')
    return p_value

# Apply two-proportion Z-test to all genes
data['Z_Test_P_Value'] = data.apply(two_proportion_z_test, axis=1)

# Apply Bonferroni correction for Z-test
data['Z_Test_Adjusted_P_Value'] = multipletests(data['Z_Test_P_Value'], method='bonferroni')[1]

# Identify significant genes after Z-test correction
data['Z_Test_Significant'] = data['Z_Test_Adjusted_P_Value'] < 0.05

# Sort by adjusted p-value
data = data.sort_values(by='Z_Test_Adjusted_P_Value')
data.head(10)

In [ ]:
sig_ethnicity = data[data['Z_Test_Significant'] == True]['Gene'].tolist()

### Gene-by-Race Comparison

For each gene, carrier and non-carrier counts are assembled across the six included race categories and evaluated with a chi-square test. P values are Bonferroni-adjusted across genes.


In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests

# Define racial groups (column names in dataset)
race_categories = ["white", "asian", "black_African", "middle_eastern", "hawaiian", "american_indian"]

# Create an empty list to store results
results = []

# Perform Chi-Square test for each gene across all races
for gene in counts_by_genes_df['gene_symbol']:
    # Extract carrier counts for each race
    carriers = [counts_by_genes_df.loc[counts_by_genes_df['gene_symbol'] == gene, f"{race}_count"].values[0] for race in race_categories]
    
    # Extract total population for each race
    total_population_counts = [total_population[race] for race in race_categories]
    
    # Compute non-carriers (Total - Carriers)
    non_carriers = [total_population_counts[i] - carriers[i] for i in range(len(race_categories))]

    # Create 2xN contingency table (where N is the number of races)
    observed = np.array([carriers, non_carriers])

    # Perform Chi-Square test
    chi2_stat, p_value, dof, expected = stats.chi2_contingency(observed)

    # Store results
    results.append([gene, p_value])

# Convert results into a DataFrame
race_comparison_df = pd.DataFrame(results, columns=["Gene", "P_Value"])

# Apply Bonferroni correction for multiple testing
race_comparison_df["Adjusted_P_Value"] = multipletests(race_comparison_df["P_Value"], method="bonferroni")[1]

# Identify significant differences after correction
race_comparison_df["Significant"] = race_comparison_df["Adjusted_P_Value"] < 0.05

# Sort by adjusted p-value
race_comparison_df = race_comparison_df.sort_values(by="Adjusted_P_Value")


In [ ]:
sig_race = race_comparison_df[race_comparison_df['Significant'] == True]['Gene'].tolist()

### Gene-by-Age Analysis

In [ ]:
# import pandas as pd
# import numpy as np
# import statsmodels.api as sm
# import statsmodels.formula.api as smf
# import scipy.stats as stats
# from statsmodels.stats.multitest import multipletests
# from datetime import datetime

# # Convert date_of_birth to Age
# current_year = datetime.now().year
# metadata['Age'] = metadata['metadata.date_of_birth'].apply(lambda x: current_year - pd.to_datetime(x).year)

# # Merge Carrier Data with Age Data
# merged_df = counts_by_genes_df.merge(metadata[['s', 'Age']], on='s', how='left')

# # Create a DataFrame to store ANOVA results
# anova_results = []

# # Perform ANOVA for each gene
# for gene in counts_by_genes_df['gene_symbol'].unique():
#     # Subset data for the given gene
#     gene_data = merged_df[merged_df['gene_symbol'] == gene]

#     # Ensure there are at least two unique groups (carrier vs non-carrier)
#     if gene_data['has_variant'].nunique() > 1:
#         try:
#             # Perform ANOVA
#             model = smf.ols('Age ~ C(has_variant)', data=gene_data).fit()
#             anova_table = sm.stats.anova_lm(model, typ=2)
#             p_value = anova_table["PR(>F)"][0]  # Extract p-value

#             # Store results
#             anova_results.append([gene, p_value])
#         except Exception as e:
#             print(f"Skipping {gene} due to error: {e}")

# # Check if there are any results before applying multiple testing correction
# if anova_results:
#     # Convert to DataFrame
#     anova_df = pd.DataFrame(anova_results, columns=["Gene", "P_Value"])

#     # Apply Bonferroni Correction
#     anova_df["Adjusted_P_Value"] = multipletests(anova_df["P_Value"], method="bonferroni")[1]

#     # Identify Significant Genes
#     anova_df["Significant"] = anova_df["Adjusted_P_Value"] < 0.05

#     # Sort by Adjusted P-Value
#     anova_df = anova_df.sort_values(by="Adjusted_P_Value")

#     # Display Results
#     import ace_tools as tools
#     tools.display_dataframe_to_user(name="ANOVA Test for Carrier Frequency by Age", dataframe=anova_df)
# else:
#     print("No valid genes for ANOVA analysis.")


#  Visualization

This section generates the figures and intermediate plotting tables used to visualize carrier counts and percentages overall and across demographic groups. 

## Gene-Level Carrier Frequency

The first set of plots summarizes the number and percentage of unique participants carrying a qualifying variant in each gene.


### Unique Carrier Count per Gene

Genes are ordered by the number of unique participants carrying at least one qualifying variant in that gene.


In [ ]:
# Sort the DataFrame in descending order of 'unique_individual_count'
counts_by_genes_df = counts_by_genes_df.sort_values(by='unique_individual_count', ascending=False)

# Plot the unique individual counts per gene
plt.figure(figsize=(12, 18))
sns.barplot(data=counts_by_genes_df, x='unique_individual_count', y='gene_symbol', orient='h')
plt.xlabel('Number of Unique Individuals with Variant')
plt.ylabel('Gene Symbol')
plt.title('Number of Individuals with Variant in Each Gene')

# Save the plot to a file
# plt.savefig('unique_individual_counts_per_gene.png', dpi=900, bbox_inches='tight')

# Show the plot
plt.show()

### Percentage of the WGS Population per Gene

Gene-level unique carrier counts are converted to percentages using the study's WGS denominator of **414,830 participants** and displayed as a horizontal bar plot.


In [ ]:
total_individual = 414830

In [ ]:
# Calculate the percentage of individuals with a variant in each gene
counts_by_genes_df['percentage'] = (counts_by_genes_df['unique_individual_count'] / total_individual) * 100
counts_by_genes_df = counts_by_genes_df.sort_values(by='percentage', ascending=False)

# Plot the percentage of individuals with a variant in each gene
plt.figure(figsize=(12, 18))
sns.barplot(data=counts_by_genes_df, x='percentage', y='gene_symbol', orient='h')
plt.xlabel('Percentage of Individuals with Variant (%)')
plt.ylabel('Gene Symbol')
plt.title('Percentage of Individuals with Variant in Each Gene')
# Save the plot to a file
# plt.savefig('unique_individual_counts_per_gene_percentage.png', dpi=900, bbox_inches='tight')
plt.show()

In [ ]:
counts_by_genes_df.head()

In [ ]:
counts_by_genes_df[counts_by_genes_df['gene_symbol'].isin(['BRCA1', 'BRCA2'])]

## Gender-Stratified Gene and Variant Summaries

The following plots compare carrier counts and percentages by gender at the gene level and then examine individual variants for a selected gene of interest.


### Gene-Level Counts by Gender

The gene-level carrier table is reshaped into long format and plotted separately for male and female participants.


In [ ]:
# Reshape the overall counts DataFrame for gender counts
overall_df_gender = counts_by_genes_df.melt(
    id_vars=["gene_symbol"], 
    value_vars=["male_count", "female_count"], 
    var_name="gender", value_name="count"
)

# Replace column names for better labels
overall_df_gender['gender'] = overall_df_gender['gender'].replace({
    'male_count': 'Male', 
    'female_count': 'Female'
})

# Plot the results in a horizontal bar plot
plt.figure(figsize=(12, 18))
sns.barplot(data=overall_df_gender, y='gene_symbol', x='count', hue='gender', orient='h', errorbar=None)
plt.xlabel('Count')
plt.ylabel('Gene Symbol')
plt.title('Overall Variant Counts per Gene by Gender')
plt.legend(title='Gender')
plt.savefig('gender_per_gene.png', dpi=900, bbox_inches='tight')

plt.show()

### Gene-Level Carrier Percentages by Gender

Counts are converted to percentages using the corresponding demographic denominator. The code also generates a publication-oriented table in which small cells are masked according to the implemented display rule.


In [ ]:
overall_df_gender['percentage'] = overall_df_gender.apply(lambda row: calculate_percentage(row, 'gender'), axis=1)

# Plot for Gender
plt.figure(figsize=(12, 18))
sns.barplot(data=overall_df_gender, y='gene_symbol', x='percentage', hue='gender', orient='h')
plt.xlabel('Percentage (%)')
plt.ylabel('Gene Symbol')
plt.title('Percentage of Variants per Gene by Gender')
plt.legend(title='Gender')
# plt.savefig('gender_per_gene_percentage.png', dpi=900, bbox_inches='tight')

plt.show()

In [ ]:
overall_df_gender['count_percentage'] = overall_df_gender.apply(
    lambda row: (
        f"{int(row['count'])} ({row['percentage']:.2f})"
        if row['count'] == 0 or row['count'] >= 20 else '*'
    ), axis=1
)

overall_df_gender_pivot = overall_df_gender.pivot(
    index="gene_symbol", 
    columns="gender", 
    values="count_percentage"
).reset_index()


overall_df_gender_pivot.head()
overall_df_gender_pivot.to_csv('overall_df_gender_pivot.csv', index=False)
!gsutil -m cp "overall_df_gender_pivot.csv" {bucket}/data/

In [ ]:
overall_df_gender[overall_df_gender['gene_symbol'] == 'TP53']

### Variant-Level Counts for a Gene of Interest

The notebook uses **MUTYH** as the gene of interest in the executed example. Individual qualifying variants are summarized by gender, and a selected locus can be inspected directly.


In [ ]:
# Define the gene of interest (e.g., AIP)
gene_of_interest = 'MUTYH'

# Filter the dataframe to only include the gene of interest
variant_df_gene = variant_counts_df[variant_counts_df['gene_symbol'] == gene_of_interest]
variant_df_gene

In [ ]:
# Reshape the DataFrame for seaborn plotting (gender variant counts)
variant_df_gender = variant_df_gene.melt(
    id_vars=["vid", "locus"], 
    value_vars=["male_count", "female_count"], 
    var_name="gender", value_name="count"
)

In [ ]:
variant_df_gender

In [ ]:
# Plot the results in a horizontal bar plot (for gender counts by variant)
plt.figure(figsize=(12, 18))
sns.barplot(data=variant_df_gender, y='vid', x='count', hue='gender', orient='h')
plt.xlabel('Count')
plt.ylabel('Variant (Locus)')
plt.title(f'Count of Variants in {gene_of_interest} by Gender')
plt.legend(title='Gender')
plt.savefig('MUTYH_gender_per_gene.png', dpi=900, bbox_inches='tight')

plt.show()

In [ ]:
# Select the locus interested in
variant_df_gender[variant_df_gender['locus'] == 'chr1:45331556']

## Ethnicity-Stratified Gene and Variant Summaries

The following plots compare carrier counts and percentages by ethnicity and provide a variant-level view for the selected gene of interest.


### Gene-Level Counts by Ethnicity

Gene-level carrier counts are plotted for Hispanic/Latino and non-Hispanic/Latino participants.


In [ ]:
# Reshape the overall counts DataFrame for ethnicity counts
overall_df_ethnicity = counts_by_genes_df.melt(
    id_vars=["gene_symbol"], 
    value_vars=["hispanic_count", "non_hispanic_count"], 
    var_name="ethnicity", value_name="count"
)

# Replace column names for better labels
overall_df_ethnicity['ethnicity'] = overall_df_ethnicity['ethnicity'].replace({
    'hispanic_count': 'Hispanic', 
    'non_hispanic_count': 'Non-Hispanic'
})

# Plot the results in a horizontal bar plot
plt.figure(figsize=(12, 18))
sns.barplot(data=overall_df_ethnicity, y='gene_symbol', x='count', hue='ethnicity', orient='h')
plt.xlabel('Count')
plt.ylabel('Gene Symbol')
plt.title('Overall Variant Counts per Gene by Ethnicity')
plt.legend(title='Ethnicity')
plt.savefig('ethnicity_per_gene.png', dpi=900, bbox_inches='tight')

plt.show()


### Gene-Level Carrier Percentages by Ethnicity

Carrier counts are normalized by the corresponding ethnicity-specific population denominator. A publication-oriented pivot table is also produced using the same small-cell masking rule implemented in the original analysis.


In [ ]:
overall_df_ethnicity['percentage'] = overall_df_ethnicity.apply(lambda row: calculate_percentage(row, 'ethnicity'), axis=1)

# Plot for Ethnicity
plt.figure(figsize=(12, 18))
sns.barplot(data=overall_df_ethnicity, y='gene_symbol', x='percentage', hue='ethnicity', orient='h')
plt.xlabel('Percentage (%)')
plt.ylabel('Gene Symbol')
plt.title('Percentage of Variants per Gene by Ethnicity')
plt.legend(title='Ethnicity')
# plt.savefig('ethnicity_per_gene_percentage.png', dpi=900, bbox_inches='tight')

plt.show()

In [ ]:
overall_df_ethnicity['count_percentage'] = overall_df_ethnicity.apply(
    lambda row: (
        f"{int(row['count'])} ({row['percentage']:.2f})"
        if row['count'] == 0 or row['count'] >= 20 else '*'
    ), axis=1
)

overall_df_ethnicity_pivot = overall_df_ethnicity.pivot(
    index="gene_symbol", 
    columns="ethnicity", 
    values="count_percentage"
).reset_index()


overall_df_ethnicity_pivot.head()
overall_df_ethnicity_pivot.to_csv('overall_df_ethnicity_pivot.csv', index=False)
!gsutil -m cp "overall_df_ethnicity_pivot.csv" {bucket}/data/

In [ ]:
sig_ethnicity

In [ ]:
overall_df_ethnicity[overall_df_ethnicity['gene_symbol'].isin(sig_ethnicity)].sort_values(by='gene_symbol')

### Variant-Level Counts for a Gene of Interest

For the executed **MUTYH** example, individual variants are summarized by ethnicity and visualized in a horizontal bar plot.


In [ ]:
# Define the gene of interest (e.g., AIP)
gene_of_interest = 'MUTYH'

# Filter the dataframe to only include the gene of interest
variant_df_gene = variant_counts_df[variant_counts_df['gene_symbol'] == gene_of_interest]

In [ ]:
# Reshape the DataFrame for seaborn plotting (ethnicity variant counts)
variant_df_ethnicity = variant_df_gene.melt(
    id_vars=["vid", "locus"], 
    value_vars=["hispanic_count", "non_hispanic_count"], 
    var_name="ethnicity", value_name="count"
)

In [ ]:
variant_df_ethnicity

In [ ]:
# Plot the results in a horizontal bar plot (for ethnicity counts by variant)
plt.figure(figsize=(12, 18))
sns.barplot(data=variant_df_ethnicity, y='vid', x='count', hue='ethnicity', orient='h')
plt.xlabel('Count')
plt.ylabel('Variant (Locus)')
plt.title(f'Count of Variants in {gene_of_interest} by Ethnicity')
plt.legend(title='Ethnicity')
plt.savefig('MUTYH_ethnicity_per_gene.png', dpi=900, bbox_inches='tight')

plt.show()

In [ ]:
# Select the locus interested in
variant_df_ethnicity[variant_df_ethnicity['locus'] == 'chr1:45331556']

## Race-Stratified Gene and Variant Summaries

The following analyses visualize gene-level carrier counts and percentages across the six race categories included in the statistical analysis and then examine individual variants within the selected gene of interest.


### Gene-Level Counts by Race

The gene-level carrier table is reshaped across the six race categories and plotted as demographic-specific counts.


In [ ]:
counts_by_genes_df

In [ ]:
# Reshape the overall counts DataFrame for race counts
overall_df_race = counts_by_genes_df.melt(
    id_vars=["gene_symbol"], 
    value_vars=["black_African_count", "white_count", "asian_count",
               "middle_eastern_count", "hawaiian_count", "american_indian_count"], 
    var_name="race", value_name="count"
)

# Replace column names for better labels
overall_df_race['race'] = overall_df_race['race'].replace({
    'black_African_count': 'Black or African American', 
    'white_count': 'White',
    'asian_count': 'Asian',
    "middle_eastern_count": 'Middle Eastern or North African',
    "hawaiian_count": 'Native Hawaiian or Other Pacific Islander',
    "american_indian_count": 'American Indian or Alaska Native'
})

# Plot the results in a horizontal bar plot
plt.figure(figsize=(12, 18))
sns.barplot(data=overall_df_race, y='gene_symbol', x='count', hue='race', orient='h')
plt.xlabel('Count')
plt.ylabel('Gene Symbol')
plt.title('Overall Variant Counts per Gene by Race')
plt.legend(title='Race')
plt.savefig('race_per_gene.png', dpi=900, bbox_inches='tight')

plt.show()


### Gene-Level Carrier Percentages by Race

Race-specific counts are normalized by the corresponding population denominators. The resulting long-format table and a masked pivot table are exported for downstream reporting.


In [ ]:
overall_df_race['percentage'] = overall_df_race.apply(lambda row: calculate_percentage(row, 'race'), axis=1)

# Plot for Race
plt.figure(figsize=(12, 18))
sns.barplot(data=overall_df_race, y='gene_symbol', x='percentage', hue='race', orient='h')
plt.xlabel('Percentage (%)')
plt.ylabel('Gene Symbol')
plt.title('Percentage of Variants per Gene by Race')
plt.legend(title='Race')
# plt.savefig('race_per_gene_percentage.png', dpi=900, bbox_inches='tight')

plt.show()

In [ ]:
overall_df_race.to_csv('overall_df_race.csv', index=False)
!gsutil -m cp "overall_df_race.csv" {bucket}/data/

In [ ]:
overall_df_race['count_percentage'] = overall_df_race.apply(
    lambda row: (
        f"{int(row['count'])} ({row['percentage']:.2f})"
        if row['count'] == 0 or row['count'] >= 20 else '*'
    ), axis=1
)

overall_df_race_pivot = overall_df_race.pivot(
    index="gene_symbol", 
    columns="race", 
    values="count_percentage"
).reset_index()


overall_df_race_pivot.head()
overall_df_race_pivot.to_csv('overall_df_race_pivot.csv', index=False)
!gsutil -m cp "overall_df_race_pivot.csv" {bucket}/data/

In [ ]:
overall_df_race_pivot.head()

In [ ]:
overall_df_race[overall_df_race['gene_symbol'].isin(['MUTYH'])].sort_values(by='percentage', ascending=False).head(10)

In [ ]:
sig_race

### Variant-Level Counts for a Gene of Interest

For the executed **MUTYH** example, individual qualifying variants are summarized and visualized across race categories.


In [ ]:
# Define the gene of interest (e.g., AIP)
gene_of_interest = 'MUTYH'

# Filter the dataframe to only include the gene of interest
variant_df_gene = variant_counts_df[variant_counts_df['gene_symbol'] == gene_of_interest]

In [ ]:
# Reshape the DataFrame for seaborn plotting (race variant counts)
variant_df_race = variant_df_gene.melt(
    id_vars=["vid", "locus"], 
    value_vars=["black_African_count", "white_count", "asian_count", "middle_eastern_count", "hawaiian_count", "american_indian_count"], 
    var_name="race", value_name="count"
)

In [ ]:
variant_df_race

In [ ]:
# Plot the results in a horizontal bar plot (for race counts by variant)
plt.figure(figsize=(12, 18))
sns.barplot(data=variant_df_race, y='vid', x='count', hue='race', orient='h')
plt.xlabel('Count')
plt.ylabel('Variant (Locus)')
plt.title(f'Count of Variants in {gene_of_interest} by Race')
plt.legend(title='Race')
plt.savefig('MUTYH_race_per_gene.png', dpi=900, bbox_inches='tight')

plt.show()

In [ ]:
# Select the locus interested in
variant_df_race[variant_df_race['locus'] == 'chr1:45331556']

## Sex-at-Birth–Stratified Gene and Variant Summaries

The following plots summarize carrier counts and percentages by sex assigned at birth and provide a variant-level view for the selected gene of interest.


### Gene-Level Counts by Sex Assigned at Birth

Gene-level carrier counts are displayed separately for participants recorded as male at birth and female at birth.


In [ ]:
# Reshape the overall counts DataFrame for sex-at-birth counts
overall_df_sex_at_birth = counts_by_genes_df.melt(
    id_vars=["gene_symbol"], 
    value_vars=["sex_at_birth_male_count", "sex_at_birth_female_count"], 
    var_name="sex_at_birth", value_name="count"
)

# Replace column names for better labels
overall_df_sex_at_birth['sex_at_birth'] = overall_df_sex_at_birth['sex_at_birth'].replace({
    'sex_at_birth_male_count': 'Male at Birth', 
    'sex_at_birth_female_count': 'Female at Birth'
})

# Plot the results in a horizontal bar plot
plt.figure(figsize=(12, 18))
sns.barplot(data=overall_df_sex_at_birth, y='gene_symbol', x='count', hue='sex_at_birth', orient='h')
plt.xlabel('Count')
plt.ylabel('Gene Symbol')
plt.title('Overall Variant Counts per Gene by Sex at Birth')
plt.legend(title='Sex at Birth')
plt.savefig('sex_birth_per_gene.png', dpi=900, bbox_inches='tight')

plt.show()

### Gene-Level Carrier Percentages by Sex Assigned at Birth

Carrier counts are normalized by the corresponding sex-at-birth denominator and visualized as percentages.


In [ ]:
overall_df_sex_at_birth['percentage'] = overall_df_sex_at_birth.apply(lambda row: calculate_percentage(row, 'sex_at_birth'), axis=1)

# Plot for Sex at Birth
plt.figure(figsize=(12, 18))
sns.barplot(data=overall_df_sex_at_birth, y='gene_symbol', x='percentage', hue='sex_at_birth', orient='h')
plt.xlabel('Percentage (%)')
plt.ylabel('Gene Symbol')
plt.title('Percentage of Variants per Gene by Sex at Birth')
plt.legend(title='Sex at Birth')
plt.savefig('sex_birth_per_gene_percentage.png', dpi=900, bbox_inches='tight')

plt.show()


### Variant-Level Counts for a Gene of Interest

For the executed **MUTYH** example, individual qualifying variants are summarized by sex assigned at birth.


In [ ]:
# Define the gene of interest (e.g., AIP)
gene_of_interest = 'MUTYH'

# Filter the dataframe to only include the gene of interest
variant_df_gene = variant_counts_df[variant_counts_df['gene_symbol'] == gene_of_interest]

In [ ]:
# Reshape the DataFrame for seaborn plotting (sex_at_birth variant counts)
variant_df_sex_at_birth = variant_df_gene.melt(
    id_vars=["vid", "locus"], 
    value_vars=["sex_at_birth_male_count", "sex_at_birth_female_count"], 
    var_name="sex_at_birth", value_name="count"
)

In [ ]:
variant_df_sex_at_birth

In [ ]:
# Plot the results in a horizontal bar plot (for sex_at_birth counts by variant)
plt.figure(figsize=(12, 18))
sns.barplot(data=variant_df_sex_at_birth, y='vid', x='count', hue='sex_at_birth', orient='h')
plt.xlabel('Count')
plt.ylabel('Variant (Locus)')
plt.title(f'Count of Variants in {gene_of_interest} by Sex-at-Birth')
plt.legend(title='Sex-at-Birth')
plt.savefig('MUTYH_sex_birth_per_gene.png', dpi=900, bbox_inches='tight')

plt.show()

In [ ]:
# Select the locus interested in
variant_df_sex_at_birth[variant_df_sex_at_birth['locus'] == 'chr1:45331556']

## Combined Multi-Panel Figure

The final visualization combines overall gene-level carrier percentages with gender-, ethnicity-, and race-stratified percentages in a 2 × 2 layout. A second version restricts the displayed genes to the subset selected by the original plotting code and saves the resulting figure for downstream use.


In [ ]:

# Create a 2-by-2 grid for subplots
fig, axes = plt.subplots(2, 2, figsize=(15, 18))

# Darker palettes
gender_palette = sns.color_palette("Dark2")
ethnicity_palette = sns.color_palette("dark")
race_palette = sns.color_palette("Dark2")
variant_color = "steelblue"  # Darker single color for non-hue plot

# Plot 1: Percentage of Individuals with Variant in Each Gene
sns.barplot(data=counts_by_genes_df, x='percentage', y='gene_symbol', orient='h', ax=axes[0, 0], color=variant_color)
# axes[0, 0].set_title('Variants per Gene (%)', fontsize=16, color='black')
axes[0, 0].set_xlabel('')
axes[0, 0].set_ylabel('', fontsize=14)
axes[0, 0].tick_params(axis='y', labelsize=12, labelcolor='black')

# Plot 2: Percentage by Gender
sns.barplot(data=overall_df_gender, y='gene_symbol', x='percentage', hue='gender', orient='h', ax=axes[0, 1], palette=gender_palette)
# axes[0, 1].set_title('By Gender (%)', fontsize=16, color='black')
axes[0, 1].set_xlabel('')
axes[0, 1].set_ylabel('', fontsize=14)
axes[0, 1].tick_params(axis='y', labelsize=12, labelcolor='black')
axes[0, 1].legend(title='Gender', fontsize=10, title_fontsize=14)

# Plot 3: Percentage by Ethnicity
sns.barplot(data=overall_df_ethnicity, y='gene_symbol', x='percentage', hue='ethnicity', orient='h', ax=axes[1, 0], palette=ethnicity_palette)
# axes[1, 0].set_title('By Ethnicity (%)', fontsize=16, color='black')
axes[1, 0].set_xlabel('')
axes[1, 0].set_ylabel('', fontsize=14)
axes[1, 0].tick_params(axis='y', labelsize=12, labelcolor='black')
axes[1, 0].legend(title='Ethnicity', fontsize=10, title_fontsize=14)

# Plot 4: Percentage by Race
grouped_data = overall_df_race.groupby(['gene_symbol', 'race'])['percentage'].sum().unstack()
grouped_data = grouped_data.loc[grouped_data.sum(axis=1).sort_values(ascending=True).index]  # Sort rows by total percentage
grouped_data.plot(kind='barh', stacked=True, ax=axes[1, 1], colormap='Dark2')
# axes[1, 1].set_title('By Race (%)', fontsize=16, color='black')
axes[1, 1].set_xlabel('')
axes[1, 1].set_ylabel('', fontsize=14)
axes[1, 1].tick_params(axis='y', labelsize=12, labelcolor='black')
axes[1, 1].legend(title='Race', fontsize=10, title_fontsize=14)

# Shared x-axis label
fig.text(0.5, 0.04, 'Percentage (%)', ha='center', fontsize=18, color='black')

# Adjust layout
plt.tight_layout(rect=[0, 0.05, 1, 1])  # Leave space for shared x-axis label

# Save the combined figure
# plt.savefig('combined_plots_updated.png', dpi=900, bbox_inches='tight')

# Show the combined figure
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

counts_by_genes_df = counts_by_genes_df.reset_index(drop=True)

# Get the index of the last occurrence of 'DICER1' in the sorted datasets
last_gene_index = counts_by_genes_df[counts_by_genes_df['gene_symbol'] == 'EGFR'].index[0]

# Filter each dataset to include all rows up to and including 'DICER1'
counts_filtered = counts_by_genes_df.iloc[:last_gene_index + 1]
gender_filtered = overall_df_gender[overall_df_gender['gene_symbol'].isin(counts_filtered['gene_symbol'])]
ethnicity_filtered = overall_df_ethnicity[overall_df_ethnicity['gene_symbol'].isin(counts_filtered['gene_symbol'])]
race_filtered = overall_df_race[overall_df_race['gene_symbol'].isin(counts_filtered['gene_symbol'])]

# Create a 2-by-2 grid for subplots
fig, axes = plt.subplots(2, 2, figsize=(11, 15))

# Plot 1: Percentage of Individuals with Variant in Each Gene
sns.barplot(data=counts_filtered, x='percentage', y='gene_symbol', orient='h', ax=axes[0, 0], color='steelblue')
# axes[0, 0].set_title('Variants per Gene (%)', fontsize=16, color='black')
axes[0, 0].set_xlabel('')
axes[0, 0].set_ylabel('', fontsize=14)
axes[0, 0].tick_params(axis='y', labelsize=12, labelcolor='black')

# Plot 2: Percentage by Gender
sns.barplot(data=gender_filtered, y='gene_symbol', x='percentage', hue='gender', orient='h', ax=axes[0, 1], palette='Dark2')
# axes[0, 1].set_title('By Gender (%)', fontsize=16, color='black')
axes[0, 1].set_xlabel('')
axes[0, 1].set_ylabel('', fontsize=14)
axes[0, 1].tick_params(axis='y', labelsize=12, labelcolor='black')
axes[0, 1].legend(title='Sex', fontsize=10, title_fontsize=14)

# Plot 3: Percentage by Ethnicity
sns.barplot(data=ethnicity_filtered, y='gene_symbol', x='percentage', hue='ethnicity', orient='h', ax=axes[1, 0], palette='dark')
# axes[1, 0].set_title('By Ethnicity (%)', fontsize=16, color='black')
axes[1, 0].set_xlabel('')
axes[1, 0].set_ylabel('', fontsize=14)
axes[1, 0].tick_params(axis='y', labelsize=12, labelcolor='black')
axes[1, 0].legend(title='Ethnicity', fontsize=10, title_fontsize=14)

# Plot 4: Percentage by Race
grouped_data = race_filtered.groupby(['gene_symbol', 'race'])['percentage'].sum().unstack()
grouped_data = grouped_data.loc[grouped_data.sum(axis=1).sort_values(ascending=True).index]  # Sort rows by total percentage
grouped_data.plot(kind='barh', stacked=True, ax=axes[1, 1], colormap='Dark2')
# axes[1, 1].set_title('By Race (%)', fontsize=16, color='black')
axes[1, 1].set_xlabel('')
axes[1, 1].set_ylabel('', fontsize=14)
axes[1, 1].tick_params(axis='y', labelsize=12, labelcolor='black')
axes[1, 1].legend(title='Race', fontsize=10, title_fontsize=14)



# # Plot 4: Percentage by Race
# sns.barplot(data=race_filtered, y='gene_symbol', x='percentage', hue='race', orient='h', ax=axes[1, 1], palette='dark')
# axes[1, 1].set_title('By Race (%)', fontsize=16, color='black')
# axes[1, 1].set_ylabel('', fontsize=14)
# axes[1, 1].tick_params(axis='y', labelsize=12, labelcolor='black')
# axes[1, 1].legend(title='Race', fontsize=12, title_fontsize=14)

# Shared x-axis label
fig.text(0.5, 0.04, 'Percentage (%)', ha='center', fontsize=18, color='black')

# Adjust layout
plt.tight_layout(rect=[0, 0.05, 1, 1])  # Leave space for shared x-axis label

# Save the combined figure
plt.savefig('filtered_combined_plots.jpg', dpi=900, bbox_inches='tight')

# Show the combined figure
plt.show()


In [ ]:
ethnicity_filtered[ethnicity_filtered['gene_symbol'] == 'MUTYH']